In [2]:
import os
import pandas as pd
import subprocess
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from pybedtools import BedTool
import pyBigWig

In [23]:
import sys
import pandas
import scipy
import statsmodels
import pybedtools
import pyBigWig

print("Python:", sys.version)
print("pandas:", pandas.__version__)
print("scipy:", scipy.__version__)
print("statsmodels:", statsmodels.__version__)
print("pybedtools:", pybedtools.__version__)
print("pyBigWig:", pyBigWig.__version__)

Python: 3.10.11 (main, Apr 20 2023, 19:02:41) [GCC 11.2.0]
pandas: 2.3.3
scipy: 1.15.3
statsmodels: 0.14.6
pybedtools: 0.12.0
pyBigWig: 0.3.18


### Step 1: Prepare BED files for noncoding de novo mutations

In [ ]:
"""
Before running CellPathway, you should prepare noncoding de novo mutations
for all patients and annotate them with CADD scores.

Input requirements:
1. Whole-genome sequencing (WGS) data in VCF format
2. Noncoding de novo variants only
3. CADD PHRED score annotated for each variant

Output:
- A tab-delimited file containing chr, pos, ref, alt, and CADD score
"""

In [3]:
cadd_list=[0,10,15,20] #set cadd threshold

for i in cadd_list:
    gt=pd.read_csv('example/autism_dnm.txt', sep='\t')
    gt=gt[gt['CADD_PHRED']>=i]
    dnm_bed=gt[['Chr','Start','End']]
    dnm_bed.to_csv(f'example/autism_dnm_{i}.bed',sep='\t',index=False,header=False)
    print(f'autism dnm count cadd={i}: {len(dnm_bed)}')

autism dnm count cadd=0: 250869
autism dnm count cadd=10: 16160
autism dnm count cadd=15: 5497
autism dnm count cadd=20: 861


### Step 2: Count total length for each enhancer

In [4]:
i=10 # set cadd threshold to 0, 10,15,20

bed_dir = "data/Atlas"
dnm_dir= "example"
overlap_dir = f"example/dnm_enhc_overlap_cadd_{i}"
out_dir = f"example/dnm_enhc_final_cadd_{i}"

# Create both directories (if not already exist)
os.makedirs(overlap_dir, exist_ok=True)
os.makedirs(out_dir, exist_ok=True)

In [5]:
out_csv = os.path.join(out_dir, "Enhc_len_atlas.csv")
results = []
for fname in os.listdir(bed_dir):
    if not fname.endswith("hg38.bed"):
        continue
    path = os.path.join(bed_dir, fname)
    total = 0
    with open(path) as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                chrom, start, end, *rest = line.strip().split()
                total += int(end) - int(start)
    
    # remove unwanted suffixes like ".hg38.bed"
    clean_name = fname.replace(".hg38.bed", "")
    #save results
    results.append({"cell": clean_name, "Enhc_bp": total})

df = pd.DataFrame(results)
df.to_csv(out_csv, index=False, sep='\t')
print(f"Saved: {out_csv}")

Saved: example/dnm_enhc_final_cadd_10/Enhc_len_atlas.csv


In [6]:
#get cell type/tissue list

bed_dir = "data/Atlas"  #/path/to/your/folder/with enhancer bed files

cell_list = [
    fname.replace(".hg38.bed", "")
    for fname in os.listdir(bed_dir)
    if fname.endswith(".hg38.bed")
]

print(cell_list)

['CD19+', 'SkMC', 'Osteoblast', 'Liver', 'Fetal_lung', 'CD14+_monocyte', 'Spleen', 'HUVEC', 'Kidney', 'B_cell_blood', 'Macrophage', 'NHBE', 'Keratinocyte', 'Pancreas', 'HCASMC', 'Fetal_brain', 'Plasma_cell_myeloma', 'reproducible_craniofacial', 'Adipocyte', 'CD8+', 'Kidney_cortex', 'CD4+', 'CD36+', 'Endometrial_stromal_cell', 'Th2', 'Trophoblast', 'CD34+', 'MSC_BM', 'Retina', 'HSC', 'NHEK', 'Fetal_kidney', 'Osteobl', 'Monocyte', 'NHLF', 'Denditric_cell', 'Fetal_heart', 'SGBS_adipocyte', 'Heart', 'Cerebellum', 'Treg_cell', 'CD20+', 'GC_B_cell', 'PrEC', 'Melanocyte', 'NHDF', 'CD14+', 'CD133+', 'NKC', 'Pancreatic_islet', 'Th1']


### Step 3: Run bedtools to get the count of overlap between dnm and enhancer regions

###### Note: Bedtools must be available in the Python environment.

In [7]:
# Run bedtools intersect for each tissue
for cell in cell_list:
    bedtools_cmd = (
        f"/mnt/isilon/wang_lab/zhangy15/miniconda3/envs/newenv/bin/bedtools intersect "
        f"-a {bed_dir}/{cell}.hg38.bed "
        f"-b {dnm_dir}/autism_dnm_{i}.bed "
        f"-c > {overlap_dir}/{cell}_dnm.bed"
    )

    try:
        result = subprocess.run(bedtools_cmd, shell=True, check=True, capture_output=True, text=True)
        print(f"[✓] {cell}: command executed successfully.")
        if result.stdout.strip():
            print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"[✗] {cell}: command failed.")
        print(e.stderr)

[✓] CD19+: command executed successfully.
[✓] SkMC: command executed successfully.
[✓] Osteoblast: command executed successfully.
[✓] Liver: command executed successfully.
[✓] Fetal_lung: command executed successfully.
[✓] CD14+_monocyte: command executed successfully.
[✓] Spleen: command executed successfully.
[✓] HUVEC: command executed successfully.
[✓] Kidney: command executed successfully.
[✓] B_cell_blood: command executed successfully.
[✓] Macrophage: command executed successfully.
[✓] NHBE: command executed successfully.
[✓] Keratinocyte: command executed successfully.
[✓] Pancreas: command executed successfully.
[✓] HCASMC: command executed successfully.
[✓] Fetal_brain: command executed successfully.
[✓] Plasma_cell_myeloma: command executed successfully.
[✓] reproducible_craniofacial: command executed successfully.
[✓] Adipocyte: command executed successfully.
[✓] CD8+: command executed successfully.
[✓] Kidney_cortex: command executed successfully.
[✓] CD4+: command execute

### Step 4: Calculate the total # of overlap dnm for each cell-specific enhancers

In [8]:
#file path to bed files
out_csv = f"{out_dir}/cell_dnm_autism.csv"

results=[]

for fname in os.listdir(overlap_dir):
    if not fname.endswith(f".bed"):
        continue

    path=os.path.join(overlap_dir, fname)
    try:
        df=pd.read_csv(path, sep='\t', header=None, comment="#", usecols=[3] ) #sum column 4
        total=df[3].sum() # sum column 4
    except Exception as e:
        print(f"Skipping {fname}:{e}")
        total=0

    #celan the fname
    fname_clean=fname.replace(f"_dnm.bed", "")
    results.append({"cell": fname_clean, f"dnm_overlap" :total})
#save to csv
out_df=pd.DataFrame(results)
out_df.to_csv(out_csv, index=False, sep='\t')
print(f"save results to {out_df}")

save results to                          cell  dnm_overlap
0                      HCASMC          811
1                      Retina          203
2                         NKC           36
3                       HUVEC          964
4                      CD133+            6
5                B_cell_blood           47
6                  Osteoblast          529
7                         HSC          207
8                       Liver          351
9                   Treg_cell          203
10                  GC_B_cell          323
11               Keratinocyte          782
12                      CD34+          693
13                 Fetal_lung          760
14                       NHLF          255
15                     MSC_BM          975
16                       NHDF          347
17                       NHBE          272
18               Fetal_kidney          392
19                Fetal_brain         1202
20                 Macrophage          641
21                      CD19+         

In [10]:
#Merge the enhc length and dnm count together
enhc_len=pd.read_csv(f'{out_dir}/Enhc_len_atlas.csv',sep='\t')
dnm_count=pd.read_csv(f"{out_dir}/cell_dnm_autism.csv",sep='\t')
enhc_dnm=pd.merge(enhc_len,dnm_count, on='cell', how='inner')
enhc_dnm.to_csv(f'{out_dir}/cell_dnm_autism_length.csv',sep='\t',index=False)

### Step 5: Merge all the bed files (hg38) for each cell type

In [11]:
# Combine all BED paths from the list
bed_files = " ".join([f"{bed_dir}/{t}.hg38.bed" for t in cell_list])

# Define Annovar command
bedtools_command = f"""
cat {bed_files} \
| sort -k1,1 -k2,2n \
| /home/zhangy15/miniconda3/envs/newenv/bin/bedtools merge -i - \
> {out_dir}/enhc_atlas_all_merged.bed

"""
# Run the Annovar command
try:
    result = subprocess.run(bedtools_command, shell=True, check=True, capture_output=True, text=True)
    print("merge command executed successfully!")
    print("Command Output:\n", result.stdout)
except subprocess.CalledProcessError as e:
    print("merge command failed!")
    print("Error Output:\n", e.stderr)

merge command executed successfully!
Command Output:
 


In [12]:
#total enhancer count
atlas_all=pd.read_csv(f'{out_dir}/enhc_atlas_all_merged.bed',sep='\t',header=None)
atlas_all[3]=atlas_all[2]-atlas_all[1]
enhc_len=atlas_all[3].sum()
enhc_len

639063306

### Step 6: Count the total # of de novo mutations within all the enhancer region

In [13]:
bedtools_command = f"""
/home/zhangy15/miniconda3/envs/newenv/bin/bedtools intersect \
-a {out_dir}/enhc_atlas_all_merged.bed \
-b {dnm_dir}/autism_dnm_{i}.bed \
-c \
> {out_dir}/cell_dnm_autism.bed

"""
# Run the Annovar command
try:
    result = subprocess.run(bedtools_command, shell=True, check=True, capture_output=True, text=True)
    print("Command executed successfully!")
    print("Command Output:\n", result.stdout)
except subprocess.CalledProcessError as e:
    print("Command failed!")
    print("Error Output:\n", e.stderr)

Command executed successfully!
Command Output:
 


In [14]:
dnm_count = {}

file_path = f'{out_dir}/cell_dnm_autism.bed'
dnm_all=pd.read_csv(file_path, sep='\t', header=None)
total = dnm_all.iloc[:, 3].sum()
dnm_count = int(total)
print(f"Total de novo count for autism: {int(total)}")

Total de novo count for autism: 5291


### Step 7: Run fisher exact test

In [15]:
# Read your data
df = pd.read_csv(f'{out_dir}/cell_dnm_autism_length.csv', sep="\t")

# Define total noncoding genome
G = enhc_len  #total number of enhancer ****

folds = []
pvals = []

for _, row in df.iterrows():
    A = row[f"dnm_overlap"]         # DNMs overlapping enhancer
    B = dnm_count-row[f"dnm_overlap"]           # DNMs in other enhancers
    C = row["Enhc_bp"]                 # enhancer bp
    D = G - C                          # enhancer bp for other enhancers

    # fold enrichment = (A/C) / (B/D)
    fold = (A / C) / (B / D)

    # Fisher's exact test
    table = [[A, C-A], [B, D-B]] # [[A, C - A], [B, D - B]]
    _, p = fisher_exact(table, alternative="greater")

    folds.append(fold)
    pvals.append(p)

df[f'dnm_overlap_other_enhancers']=dnm_count-df[f"dnm_overlap"]

# Add results to DataFrame
df[f"Fold_enrichment"] = folds
df[f"P_value"] = pvals

# Apply FDR correction (Benjamini–Hochberg)
_, fdr_p, _, _ = multipletests(df[f"P_value"], method='fdr_bh')
df[f"P_FDR"] = fdr_p

# Save results
out_path = f"{out_dir}/autism_FC_{i}.csv"
df.to_csv(out_path, sep=",", index=False)

print(f"Enrichment results saved to: {out_path}")

Enrichment results saved to: example/dnm_enhc_final_cadd_10/autism_FC_10.csv


In [16]:
#read CellPathway output files
fc=pd.read_csv('example/autism_FC_10.csv', sep=',')
fc=fc.sort_values(by='P_FDR')[:10]
fc

,cell,Enhc_bp,dnm_overlap,dnm_overlap_other_enhancers,Fold_enrichment,P_value,P_FDR
15,Fetal_brain,73799119,1202,4089,2.251581,7.621348e-116,3.886888e-114
17,reproducible_craniofacial,92007086,1350,3941,2.036751,1.009323e-99,2.573773e-98
31,Fetal_kidney,20384498,392,4899,2.428532,7.663079e-51,1.302723e-49
4,Fetal_lung,55688424,760,4531,1.757124,2.089160e-41,2.663679e-40
2,Osteoblast,39645022,529,4762,1.679607,2.559020e-26,2.610201e-25
13,Pancreas,36666954,484,4807,1.654165,3.575184e-23,3.038906e-22
28,Retina,12780649,203,5088,1.955088,1.355652e-17,8.642282e-17
45,NHDF,25934817,347,4944,1.659279,1.260939e-17,8.642282e-17
36,Fetal_heart,79162320,865,4426,1.382284,2.648341e-17,1.455639e-16
25,Trophoblast,41632826,505,4786,1.514154,2.854194e-17,1.455639e-16


### Step 8: Do the tad annotation 

In [17]:
import os
import pandas as pd
# os.environ['PATH']='/home/zhangy15/miniconda3/envs/myenv2/bin:' +os.environ['PATH']
from pybedtools import BedTool
import pyBigWig

def superset_tad(query_region, tad_r, tad_b):
    """
    Determine whether query_region is within a TAD or crosses a TAD boundary,
    and return the corresponding covering region + status.
    """
    if isinstance(query_region, (list, tuple)):
        query_region = ' '.join([str(i) for i in query_region])
    query_region_bed = BedTool(query_region, from_string=True)
    tad_r_bed = BedTool.from_dataframe(tad_r)
    tad_b_bed = BedTool.from_dataframe(tad_b)

    inter_b = tad_b_bed.intersect(query_region_bed, wa=True, wb=True).to_dataframe()

    if inter_b.shape[0] == 0:
        inter_r = tad_r_bed.intersect(query_region_bed, wa=True).to_dataframe()
        if inter_r.shape[0] == 0:
            region = [None, None, None]
            status = "outside_TAD"
        else:
            region = inter_r.values.tolist()[0]
            status = "in_TAD"
    else:
        boundary_selected = ' '.join([
            inter_b.chrom.iloc[0],
            str(min(inter_b.start)),
            str(max(inter_b.end))
        ])
        boundary_selected_bed = BedTool(boundary_selected, from_string=True)
        fu = boundary_selected_bed.closest(tad_r_bed, D='ref', fu=True).to_dataframe()
        fd = boundary_selected_bed.closest(tad_r_bed, D='ref', fd=True).to_dataframe()
        region = [fu.values[0][0], fu.values[0][4], fd.values[0][5]]
        status = "cross_TAD_boundary"

    return region, status


def tad_elements(element_bb, tad_region_tri):
    """
    Get all gene/elements within a TAD genomic region.
    """
    if None in tad_region_tri:  # skip invalid TAD regions
        return None, None, [], []

    try:
        elements_coords = element_bb.entries(
            tad_region_tri[0],
            int(tad_region_tri[1]),
            int(tad_region_tri[2])
        )
    except Exception:
        return None, None, [], []

    gene_names, gene_indicator, elements_coords_ = [], [], []
    for ele in elements_coords:
        elements_coords_.append([tad_region_tri[0]] + list(ele[:2]) + [ele[2].split('\t')[0]])
        gene_indicator_pair = ele[2].split('\t')
        gene_indicator.append(int(gene_indicator_pair[1]))
        if gene_indicator_pair[1] == '1':
            gene_names.append(gene_indicator_pair[0])

    elements_coords_df = pd.DataFrame(elements_coords_, columns=["chrom", "start", "end", "feature"])
    elements_coords_bed = BedTool.from_dataframe(elements_coords_df)
    return elements_coords_df, elements_coords_bed, gene_indicator, gene_names

In [20]:
# # Load enhancer and TAD data
Path = "data"
tad_path = os.path.join(Path, "tad_w_boundary_08.bed")
elements_path = os.path.join(Path, "genes_w_noncoding.bb")

# Load TAD BED file (format: chrom, start, end, label)
tad = pd.read_csv(tad_path, sep="\t", header=None, names=["chrom", "start", "end", "label"])
tad_r = tad.loc[tad["label"].astype(str) == "0", ["chrom", "start", "end"]].reset_index(drop=True)
tad_b = tad.loc[tad["label"].astype(str) == "1", ["chrom", "start", "end"]].reset_index(drop=True)

# Open the .bb file
element_bb = pyBigWig.open(elements_path, "r")

# Load enhancer BED file
fetal_brain = pd.read_csv(f"example/dnm_enhc_overlap_cadd_10/Fetal_brain_dnm.bed", sep="\t", header=None)
enhancers = fetal_brain[fetal_brain[3] > 0][[0, 1, 2]] # tissue that have overlap with enhancers
enhancers.columns = ["chrom", "start", "end"]

results = []

# ----------------------------
# Loop through each enhancer region
# ----------------------------
for _, row in enhancers.iterrows():
    enhc_tri = [row["chrom"], int(row["start"]), int(row["end"])]
    tad_region, status = superset_tad(enhc_tri, tad_r, tad_b)

    # Get all elements (genes) in that TAD
    elements_coords_df, elements_coords_bed, gene_indicator, gene_names = tad_elements(element_bb, tad_region)

    results.append({
        "enh_chr": row["chrom"],
        "enh_start": row["start"],
        "enh_end": row["end"],
        "tad_chr": tad_region[0],
        "tad_start": tad_region[1],
        "tad_end": tad_region[2],
        "status": status,
        "gene_names": ",".join(gene_names) if gene_names else None,
        "n_genes": len(gene_names),
#         "gene_indicator": gene_indicator if gene_indicator else None
    })

results_df = pd.DataFrame(results)

#sfari gene
sfari_genes=pd.read_csv('example/SFARI_Gene.csv',sep=',')
sfari_genes=sfari_genes['gene-symbol'].to_list()
# Convert SFARI list to uppercase for consistent matching
sfari_set = set(g.upper() for g in sfari_genes)

def get_sfari_overlap(gene_str):
    if pd.isna(gene_str) or gene_str is None:
        return ""
    genes = [g.strip().upper() for g in gene_str.split(",") if g.strip()]
    overlap = [g for g in genes if g in sfari_set]
    return ",".join(overlap)

# Apply to each row
results_df["sfari_overlap_gene"] = results_df["gene_names"].apply(get_sfari_overlap)
results_df.to_csv(f"example/tad_brain_autism.tsv", sep=",", index=False)

# Extract all non-empty overlap entries, split by comma, and flatten into one list
all_sfari_overlap_genes = (
    results_df["sfari_overlap_gene"]
    .dropna()
    .loc[results_df["sfari_overlap_gene"] != ""]
    .str.split(",")
    .sum()
)
# sort_sfari_genes = sorted(set(all_sfari_overlap_genes))
sort_sfari_genes = sorted(all_sfari_overlap_genes)
pd.Series(sort_sfari_genes).to_csv(f"example/tad_brain_autism_domain_gene.txt", index=False, header=False)

In [22]:
results_df[:10]

,enh_chr,enh_start,enh_end,tad_chr,tad_start,tad_end,status,gene_names,n_genes,sfari_overlap_gene
0,chr1,1535800,1539680,chr1,764620,3423436,in_TAD,"SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,A...",63,"SAMD11,SKI"
1,chr1,2229741,2236661,chr1,764620,3423436,in_TAD,"SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,A...",63,"SAMD11,SKI"
2,chr1,2269932,2275522,chr1,764620,3423436,in_TAD,"SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,A...",63,"SAMD11,SKI"
3,chr1,2540241,2546361,chr1,764620,3423436,in_TAD,"SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,A...",63,"SAMD11,SKI"
4,chr1,2986186,2988806,chr1,764620,3423436,in_TAD,"SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,A...",63,"SAMD11,SKI"
5,chr1,3326120,3333136,chr1,764620,3423436,in_TAD,"SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,A...",63,"SAMD11,SKI"
6,chr1,3706420,3707546,chr1,3463436,3783436,in_TAD,"ARHGEF16,MEGF6,TPRG1L,WRAP73,TP73,CCDC27,SMIM1...",8,
7,chr1,5151280,5152490,chr1,3823436,5959940,in_TAD,"CEP104,DFFB,C1orf174,AJAP1,NPHP4",5,
8,chr1,6893530,6895430,chr1,6479940,7599940,in_TAD,"PLEKHG5,NOL9,TAS1R1,ZBTB48,KLHL21,PHF13,THAP3,...",9,
9,chr1,6984670,6985530,chr1,6479940,7599940,in_TAD,"PLEKHG5,NOL9,TAS1R1,ZBTB48,KLHL21,PHF13,THAP3,...",9,
